In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from loguru import logger
import os

print("Current working directory:", os.getcwd())

# Configure loguru for notebook display
# logger.remove()
# logger.add(lambda msg: print(msg, end=''), colorize=True, format="{level.icon} {message}")

# Data paths
ANNOT_FILE = "data/dataset_092624.xlsx"
OUTPUT_FILE = "data/dataset_092624_validated.xlsx"

Current working directory: c:\Users\beav3503\dev\llm_metadata


## 1. Load Raw Data

In [3]:
# Load the raw Excel file
raw_df = pd.read_excel(ANNOT_FILE)

print(f"Loaded {len(raw_df)} rows, {len(raw_df.columns)} columns")
print(f"\nColumns: {list(raw_df.columns)}")
raw_df.head()

Loaded 418 rows, 43 columns

Columns: ['id', 'url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_articles', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'url.1', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']


,id,url,title,full_text,publication_year,source,id_query,reason_non_valid,valid_yn,dataset_relevance,...,threatened_species,new_species_science,new_species_region,bias_north_south,url.1,MC_dataset_type,MC_spatial_range,MC_temporal_range,MC_relevance,MC_relevance_modifiers
0,3,https://doi.org/10.5061/dryad.05qfttf0n,Fish mock community with 41 species from 13 or...,Fish mock community with 41 species from 13 or...,2020,zenodo,3,NaN,yes,X,...,NaN,NaN,NaN,NaN,https://doi.org/10.5061/dryad.05qfttf0n,L,X,X,X,X
1,4,https://doi.org/10.5061/dryad.0rxwdbs2c,Exploration and diet specialization in eastern...,Exploration and diet specialization in eastern...,2022,zenodo,"3,7,4,6",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.0rxwdbs2c,L,L,M,L,L
2,5,https://doi.org/10.5061/dryad.121sk,Data from: Paternity analysis of wood turtles ...,Data from: Paternity analysis of wood turtles ...,2017,zenodo,"3,8",NaN,yes,M,...,True,False,False,False,https://doi.org/10.5061/dryad.121sk,H,X,L,L,M
3,7,https://doi.org/10.5061/dryad.12jm63xtw,Data from: 60 specific eDNA qPCR assays to det...,Data from: 60 specific eDNA qPCR assays to det...,2020,zenodo,"3,8",NaN,yes,X,...,NaN,NaN,NaN,NaN,https://doi.org/10.5061/dryad.12jm63xtw,X,X,X,X,X
4,9,https://doi.org/10.5061/dryad.1771t,Data from: Resampling method for applying dens...,Data from: Resampling method for applying dens...,2016,dryad,"0,3,4,8,10",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.1771t,H,X,L,L,L


## 2. Run Validation

In [5]:
from llm_metadata.schemas import DataFrameValidator, ValidationReport

# Initialize validator (strict=False to collect all errors)
validator = DataFrameValidator(strict=False)

# Run validation
report = validator.validate(raw_df)

# Display summary
print(report.summary())

2026-01-06 12:11:52.479 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 418 valid, 0 invalid, 0 errors


VALIDATION REPORT
Total rows:     418
Valid rows:     418 (100.0%)
Invalid rows:   0

Error Breakdown:
  Type/Schema errors: 0
  Data/Value errors:  0

Errors by Field:


## 3. Review Errors

In [6]:
# Get errors as DataFrame for inspection
errors_df = report.errors_to_dataframe()

if len(errors_df) > 0:
    print(f"Total errors: {len(errors_df)}")
    print(f"\nError distribution by field:")
    print(errors_df['field'].value_counts())
    print(f"\nError distribution by type:")
    print(errors_df['error_type'].value_counts())
else:
    print("✅ No errors found!")

errors_df.head(20)

✅ No errors found!


,row_index,field,raw_value,error_type,message,suggested_fix


In [7]:
# View errors with suggested fixes
if len(errors_df) > 0:
    errors_with_fixes = errors_df[errors_df['suggested_fix'] != ''][['row_index', 'field', 'raw_value', 'suggested_fix']]
    print(f"Errors with suggested fixes: {len(errors_with_fixes)}")
    display(errors_with_fixes)

## 4. Review Invalid Rows

In [8]:
# Get invalid rows for correction
invalid_df = report.invalid_rows_to_dataframe()

# Select only relevant columns for display
relevant_cols = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
                 'temporal_range', 'temp_range_i', 'temp_range_f', 'taxons', 'referred_dataset']
available_cols = [c for c in relevant_cols if c in invalid_df.columns]

print(f"Invalid rows: {len(invalid_df)}")
if len(invalid_df) > 0:
    display(invalid_df[available_cols].head(10))

Invalid rows: 0


## 7. Export Validated Data

In [10]:
# Get validated rows as DataFrame
validated_df = report.valid_rows_to_dataframe()

print(f"Validated rows: {len(validated_df)}")
validated_df.head()

Validated rows: 418


,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,taxons,referred_dataset
0,[genetic_analysis],None,NaN,None,NaN,NaN,None,None
1,[presence-only],[administrative_units],0.2,2012-2017,2012.0,2017.0,None,None
2,[genetic_analysis],None,NaN,2006-2007,2006.0,2007.0,None,None
3,[other],None,NaN,None,NaN,NaN,None,None
4,[density],[sample],NaN,2008,2008.0,2008.0,None,None


In [11]:
# Export options:

# Option 1: Export only validated rows (schema-compliant and cleaned)
validated_df.to_excel(OUTPUT_FILE, index=False)
print(f"Exported {len(validated_df)} validated rows to {OUTPUT_FILE}")

# Option 2: Export all rows with validation status column
export_df = raw_df.copy()
# Note: validator.validate() returns a report with valid_indices
export_df['_validation_status'] = 'invalid'
export_df.loc[report.valid_indices, '_validation_status'] = 'valid'

# Uncomment to save:
# export_df.to_excel(OUTPUT_FILE, index=False)
# print(f"Exported {len(export_df)} rows with validation status to {OUTPUT_FILE}")

print("Export cells ready - uncomment desired option to save.")

Exported 418 validated rows to data/dataset_092624_validated.xlsx
Export cells ready - uncomment desired option to save.
